# Elasticsearch: Conexión y primera búsqueda

## 0. Instalación

In [1]:
!pip install elasticsearch --quiet


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 1. Credenciales de conexión

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

ES_URL        = os.getenv("ES_URL", "")
ES_USER     = os.getenv("ES_USER", "elastic")
ES_PASSWORD            = os.getenv("ES_PASSWORD", "")
ES_INDEX            = os.getenv("ES_INDEX", "workshop_docs")

print("Credenciales cargadas")

Credenciales cargadas


## 2. Conectar al cluster

In [4]:
from elasticsearch import Elasticsearch

es = Elasticsearch(
    ES_URL,
    basic_auth=(ES_USER, ES_PASSWORD),
    verify_certs=True
)

# Verificar conexión
if es.ping():
    print('Conectado al cluster de Elasticsearch')
else:
    print('No se pudo conectar. Verificá URL y credenciales.')

No se pudo conectar. Verificá URL y credenciales.


## 3. Info del cluster

In [ ]:
info = es.info()
print('Nombre del cluster:', info['cluster_name'])
print('Versión:           ', info['version']['number'])

## 4. Ver el índice del workshop

In [ ]:
# Ver los mappings del índice (la 'estructura' de los documentos)
mappings = es.indices.get_mapping(index=ES_INDEX)
props = mappings[ES_INDEX]['mappings'].get('properties', {})

print(f'Campos del índice "{ES_INDEX}":')
print('-' * 40)
for campo, config in props.items():
    tipo = config.get('type', 'object')
    dims = config.get('dims', '')
    dims_str = f' ({dims} dims)' if dims else ''
    print(f'  {campo:<20} {tipo}{dims_str}')

In [ ]:
# Ver cuántos documentos tiene el índice
count = es.count(index=ES_INDEX)
print(f'\nDocumentos indexados: {count["count"]}')

## 5. Ver un documento de ejemplo

In [ ]:
resultado = es.search(
    index=ES_INDEX,
    body={'query': {'match_all': {}}, 'size': 1}
)

doc = resultado['hits']['hits'][0]
print('ID del documento:', doc['_id'])
print()
fuente = doc['_source']
for k, v in fuente.items():
    if k == 'embedding':
        print(f'  embedding: [{v[0]:.4f}, {v[1]:.4f}, ... ] ({len(v)} dims)')
    else:
        val = str(v)[:100] + '...' if len(str(v)) > 100 else v
        print(f'  {k}: {val}')

## 6. Primera búsqueda por keyword

In [ ]:
query_texto = 'error de autenticación'

resultado = es.search(
    index=ES_INDEX,
    body={
        'query': {
            'match': {
                'contenido': {
                    'query': query_texto,
                    'operator': 'or'
                }
            }
        },
        'size': 3,
        '_source': ['contenido', 'fuente']
    }
)

hits = resultado['hits']['hits']
print(f'Query: "{query_texto}"')
print(f'Resultados: {len(hits)}')
print('-' * 50)
for i, hit in enumerate(hits, 1):
    print(f'\n[{i}] Score: {hit["_score"]:.4f}')
    print(f'    Fuente: {hit["_source"].get("fuente", "N/A")}')
    contenido = hit['_source'].get('contenido', '')[:200]
    print(f'    Contenido: {contenido}...')